In [ ]:
OPENAI_API_KEY=""

In [1]:
!pip install openai

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------ --------------- 1.0/1.7 MB 7.5 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 6.3 MB/s  0:00:00

   -------- ------------------------------- 1/5 [idna]
   -------- ------------------------------- 1/5 [idna]
   ---------------- ----------------------- 2/5 [httpcore2]
   ---------------- ----------------------- 2/5 [httpcore2]
   ---------------- ----------------------- 2/5 [httpcore2]
   ---------------- ----------------------- 2/5 [httpcore2]
   ------------------------ --------------- 3/5 [httpx2]
   ------------------------ --------------- 3/5 [httpx2]
   ------------------------ --------------- 3/5 [httpx2]
   ------------------------ --------------- 3/5 [httpx2]
   -------------------------------- ------- 4/5 [openai]
   -------------------------------- ------- 4/5 [openai]
   --------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.35.1 which is incompatible.


In [1]:
# Prompt => openai model => response

from openai import OpenAI
client = OpenAI( api_key = OPENAI_API_KEY )

response = client.responses.create(
    model="gpt-5.6",
    input="Write a one-sentence bedtime story about a unicorn.",
)

print(response.output_text)

In [ ]:
# input in the form of list

response = client.responses.create(
    model="gpt-5.6",
    input=[
        {"role":"user", "content":"a+2=7, find a"},
        {"role":"system", "content":"act like a math teacher"}
    ]
)

print(response.output_text)

In [ ]:
response = client.responses.create(
    model="gpt-5.6",
    input="a+2=7, find a"
)

print(response.output_text)

In [ ]:
response

In [ ]:
# Image

response = client.responses.create(
    model="gpt-5.6",
    input=[
        {
            "role":"user",
            "content":[
                {
                    "type" : "input_text",
                    "text" : "Which country's flag is in the image?"
                },
                {
                    "type" : "input_image",
                    "image_url" : "https://www.shutterstock.com/image-photo/highly-realistic-image-national-flag-260nw-2660354553.jpg "
                }
            ]
        }
    ]
)

print(response.output_text)

### AI Personal Assistant

In [ ]:
# queries answer
# email => summarize

class PersonalAssistant:
    def __init__(self):
        print("Hi, I am you AI assistant. How can I help you?")

    def ans_query(self):
        question = input("Ask me anything: ")
        
        response = client.responses.create(
            model="gpt-5.6",
            input=[
                { "role":"system", "content":"Act like a helpful personal assistant" },
                { "role":"user" , "content":question }
            ],
            temperature=0.7, # creative
            max_output_tokens=512
        )
        
        print(response.output_text.strip())

    def summarize_email(self):
        email_text = input("Paste your email here: ")
        prompt = f"summarize the following email in 2-3 sentences: {email_text}"

        response = client.responses.create(
            model = "gpt-5.6",
            input=[
                {"role":"system", "content":"Act like an expert email assistant"},
                {"role":"user", "content":prompt}
            ],
            temperature=0.3,
            max_output_tokens=512
            
        )

        print("\n\nSummary: ",response.output_text.strip())

In [ ]:
assistant = PersonalAssistant()

In [ ]:
assistant.ans_query()

In [ ]:
# sample email

sample_email = """"give a sample email"""

assistant.summarize_email()

## Images in OpenAI

### Responses API

In [ ]:
import base64
# images => base64 encoded strings => decode


In [ ]:
response = client.responses.create(
    model="gpt-5",
    input="Generate an image of a highly detailed futuristic spaceship, flying near nebula",
    tools=[{"type" : "image_generation"}]
)

image_data = [
    output.result
    for output in response.output
    if output.type == "image_generation_call"
]

# decode & store in a file
if image_data:
    image_base64 = image_data[0]
    with open("spaceship.png","wb") as f:
        image_bytes = base64.b64decode(image_base64)
        f.write(image_bytes)

### Image API

In [ ]:
prompt = """
A children's book drawing of a veterinarian using a stethoscope to
listen to the heartbeat of a baby otter.
"""

result = client.images.generate(
    model = "gpt-image-1.5",
    prompt = prompt
)

image_base64 = result.data[0].b64_json
image_bytes = base64.b64decode(image_base64)

# Save the image to a file
with open("otter.png","wb") as f:
    f.write(image_bytes)

In [ ]:
# EDIT image

prompt = """
Take the reference image and draw bear instead of baby otter
"""

result = client.images.edit(
    model = "gpt-image-1.5",
    prompt = prompt,
    image=[
        open("otter.png", "rb"),
    ]
)

image_base64 = result.data[0].b64_json
image_bytes = base64.b64decode(image_base64)

# Save the image to a file
with open("bear.png","wb") as f:
    f.write(image_bytes)
